In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import altair as alt
import pyarrow
from numpy.ma.extras import unique
from polars import Null
import matplotlib.pyplot as plt
import squarify
import seaborn as sns

from func.UsefulFunctions import *

This notebook documents the cleaning and inspection process for the artist data on wikidata

In [ ]:
artistDF = pl.read_parquet("./Data/pipelinesArtistsWikidata.parquet")
print(f"There are {len(artistDF["artist"].unique())} unique artist entities after the removal of the disallowed occupations.")
display(artistDF.head(3))

In [ ]:
#inspect the different fields in the system that are error prone
#see the different genders in the dataframe
print(artistDF[["artist","genderLabel"]].unique().select(pl.col("genderLabel").value_counts(sort=True)).unnest("genderLabel"))
print(artistDF[["genderLabel"]].unique().select(pl.col("genderLabel")).to_series().to_list())

#see some genders who have unknown or none gender
noGender = artistDF.filter(
    pl.col("gender").is_null()
)
noGender = noGender.select(pl.col(["artist","gender","genderLabel"])).unique()
print(noGender)
#Save the file for individuals who do not have a gender
noGender.write_csv("./missingValuesStore/artistsEntitiesWithoutGenders.csv",)


Checked the genders to ensure that error/unknown genders were successfully nulled.

In [ ]:
#Null the artist labels for the individuals who do not have an artist label

noNames = artistDF.filter(
    pl.col("artist").is_in(pl.col("artistLabel").implode())
).select(pl.col("artistLabel")).unique()
name_dict = {k: None for k in noNames["artistLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("artistLabel").replace(name_dict))

noNamesDF = artistDF.filter(
    pl.col("artistLabel").is_null(),
).select(pl.col(["artist","artistLabel"])).unique()
display(noNamesDF)
print(f"There are {len(noNamesDF)} artists with no default labels/labels in english.")
noNamesDF.write_csv("./missingValuesStore/artistsEntitiesWithoutLabels.csv")

In [ ]:
#The individuals with a country without an english label
#First let's see if there are any unknown or erroneous countries by using the Q-numbers and there are
artistDF.filter(
    pl.col('countryCitizenshipLabel').str.starts_with("Q")
).select(pl.col('countryCitizenshipLabel')).unique().to_series().to_list()

noLabel_Country = artistDF.filter(
    pl.col("countryCitizenship").is_in(pl.col("countryCitizenshipLabel").implode())
).select(pl.col("countryCitizenshipLabel")).unique()
countryName_dict = {k: None for k in noLabel_Country["countryCitizenshipLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("countryCitizenshipLabel").replace(countryName_dict))
artistDF = artistDF.with_columns(
    countryCitizenship = pl.when(~pl.col('countryCitizenship').str.starts_with("Q")).then(None).otherwise(pl.col('countryCitizenship'))
)
noCountryLabel = artistDF.filter(
    pl.col("countryCitizenshipLabel").is_null(),
).select(pl.col(["countryCitizenship","countryCitizenshipLabel"])).unique()
display(noCountryLabel)
# noCountry = artistDF.filter(
#     pl.col("countryCitizenship").is_in(pl.col("countryCitizenshipLabel").implode())
# ).select(pl.col("countryCitizenship")).unique()
# display(noCountry)
# print(f"There are {len(noCountry)} artists with a country with no labels. These will be cleaned out.")
# noCountry.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledCountries.csv")

In [ ]:
#Genres with no labels in english/default language
noGenre = artistDF.filter(
    pl.col("genre").is_in(pl.col("genreLabel").implode())
).select(pl.col('genreLabel')).unique()
noGenreName_dict = {k: None for k in noGenre["genreLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("genreLabel").replace(noGenreName_dict))
artistDF = artistDF.with_columns(
    genre = pl.when(~pl.col('genre').str.starts_with("Q")).then(None).otherwise(pl.col('genre'))
)

noGenreLabel = artistDF.filter(
    pl.col("genreLabel").is_null(),
).select(pl.col(["artist","artistLabel","genre","genreLabel"])).unique()
display(noGenreLabel)
#print(f"There are {len(noGenre)} artists with a genre with no labels. These will be cleaned out.")
noGenreLabel.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledGenresOrNoGenres.csv")

In [ ]:
#Record Labels without a label
noRecordLabelLabel = artistDF.filter(
    pl.col("recordLabel").is_in(pl.col("recordLabelLabel").implode())
).select(pl.col(["recordLabelLabel"])).unique()
recordLabel_dict = {k: None for k in noRecordLabelLabel["recordLabelLabel"].to_list()}
artistDF = artistDF.with_columns(
    pl.col("recordLabelLabel").replace(recordLabel_dict),
    recordLabel = pl.when(~pl.col('recordLabel').str.starts_with("Q")).then(None).otherwise(pl.col('recordLabel')),
)

recordLabelsWithNoLabel = artistDF.filter(
    pl.col("recordLabelLabel").is_null()
).select(["artist","artistLabel","recordLabel","recordLabelLabel"]).unique()
display(recordLabelsWithNoLabel)

recordLabelsWithNoLabel.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledRecordLabelsOrNoRecordLabels.csv")

In [ ]:
#Influences Labels without a label
noLabelsInfluencedBY = artistDF.filter(
    pl.col("influencedBY").is_in(pl.col("influencedBYLabel").implode())
).select(pl.col(["influencedBYLabel"])).unique()

influencedBName_dict = {k: None for k in noLabelsInfluencedBY["influencedBYLabel"].to_list()}
artistDF = artistDF.with_columns(
    pl.col("influencedBYLabel").replace(influencedBName_dict),
    influencedBY = pl.when(~pl.col('influencedBY').str.starts_with("Q")).then(None).otherwise(pl.col('influencedBY'))
)

entitiesWithNullInfluences = artistDF.filter(
    pl.col("influencedBYLabel").is_null()
).select(pl.col(["artist","artistLabel","influencedBY","influencedBYLabel"])).unique()
display(entitiesWithNullInfluences)
entitiesWithNullInfluences.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledInfluencesOrNoInfluences.csv")

In [ ]:
#These are checks to ensure that size of the dataset has not mistakenly been reduced
print(f"The size of the dataset now is {len(artistDF["artist"].unique())}")

In [ ]:
#Look at selection of the musicBrainz table for any anomalies
mBtable = artistDF.select(pl.col(["artist","artistLabel","musicBrainzID"])).unique()
mB_issues = mBtable.filter(
    pl.col("musicBrainzID").str.contains("http")
).unique()
display(mB_issues)


mB_issues.write_csv("./missingValuesStore/artistsEntitiesWithFormattingIssuesInTheirMusicBrainzID.csv")
mB_users = mBtable.filter(
    pl.col("musicBrainzID").str.contains("user")
)
display(mB_users)


#Because it is two people who had these errored user links in there I will drop those two rows as their info will continue to be in the dataset
artistDF = artistDF.filter(
    (~pl.col("musicBrainzID").str.contains("user")) | (pl.col("musicBrainzID").is_null())
)

#Get the artist ID of the people with the links for theirs instead of the number
artistDF = artistDF.with_columns(
    musicBrainzID = pl.when(pl.col("musicBrainzID").str.contains("artist")).then(
        pl.col("musicBrainzID").str.split("/").list.last()
    ).otherwise(pl.col("musicBrainzID")),
)

#See a selection of artists to make sure changes were done correctly.
display(artistDF.filter((pl.col("artist") == "Q138387061") | (pl.col("artist") == "Q138413725") | (pl.col("artist") == "Q139919749")  ).unique())


In [ ]:
#These are checks to ensure that size of the dataset has not mistakenly been reduced
print(f"The size of the dataset now is {len(artistDF["artist"].unique())}")

In [ ]:
#Look at the data in the statements tab to ensure they are all numbers
print(artistDF.schema)

#By checking the schema I can see that the statementCount is rightfully a series of numbers (Float64)
#Did a mean to confirm that there were no strings in there that could cause an issue
artistDF.select(pl.col("statementCount")).unique().mean()

In [ ]:
#Look at the wiki column and see that every article has the wiki link in there by checking for those without
artistDF.filter(
    (~pl.col("article").str.contains("en.wikipedia"))
).unique()

#From this I can see that all the articles in here are correct links by checking that they contain en.wikipedia

In [ ]:
artistDF = artistDF.unique()

In [ ]:
len(artistDF["artist"].unique())

In [ ]:
artistDF.write_parquet("./Data/Network_artistsWikidata.parquet")

At this point I have checked all the fields and attempted to clean up the data to highlight any unknowns. Each row should now be unique and contain a single value

In [ ]:
#Look at the gender split of the data
#Discover the gender gap in the set and the number of people who have a null/unknown gender
artistsByGender = artistDF.select(
    pl.col("artist"),
    pl.col("gender"),
    pl.col("genderLabel"),
).unique()
artistGenderGrouped = artistsByGender.group_by(["gender", "genderLabel"]).agg(pl.col("artist").count()).sort(by=pl.col("artist"),descending=True)
artistGenderGrouped.write_csv("./Visuals Data/artistsGroupedByGender.csv", null_value="null/unknown")
artistGenderGroupedFRAC = artistsByGender["genderLabel"].drop_nulls().value_counts(sort=True, normalize=True, name="fraction")
display(artistGenderGrouped)
display(artistGenderGroupedFRAC)

In [ ]:
wikipediaArtistsGender = artistDF.filter(
    pl.col("article").is_not_null()
).select(
    pl.col(["artist","gender","genderLabel"])
).unique()
print(len(wikipediaArtistsGender["artist"].unique()))
wikipediaArtistGenderGrouped = wikipediaArtistsGender.group_by(["gender", "genderLabel"]).agg(pl.col("artist").count()).sort(by=pl.col("artist"),descending=True)
wikipediaArtistGenderGroupedFRAC = wikipediaArtistsGender["genderLabel"].drop_nulls().value_counts(sort=True, normalize=True, name="fraction")
wikipediaArtistGenderGrouped.write_csv("./Visuals Data/WikipediaArtistsGroupedByGender.csv", null_value="null/unknown")
display(wikipediaArtistGenderGrouped)
display(wikipediaArtistGenderGroupedFRAC)

men_womenWiki = wikipediaArtistGenderGrouped[0:2]
print(men_womenWiki)
other_gendersWiki = wikipediaArtistGenderGrouped[2:].drop_nulls().select(pl.col("artist")).sum()
print(other_gendersWiki)
dfOthersWiki = pl.DataFrame({"gender":None,"genderLabel":"other", "artist":other_gendersWiki.item(0,0)})
genderList = men_womenWiki["genderLabel"].to_list()
genderList.extend(dfOthersWiki["genderLabel"].to_list())
print(genderList)
countListWiki = men_womenWiki["artist"].to_list()
countListWiki.extend(dfOthersWiki["artist"].to_list())


In [ ]:
men_women = artistGenderGrouped[0:2]
print(men_women)
other_genders = artistGenderGrouped[2:].drop_nulls().select(pl.col("artist")).sum()
print(other_genders)
dfOthers = pl.DataFrame({"gender":None,"genderLabel":"other", "artist":other_genders.item(0,0)})
genderList = men_women["genderLabel"].to_list()
genderList.extend(dfOthers["genderLabel"].to_list())
print(genderList)
countList = men_women["artist"].to_list()
countList.extend(dfOthers["artist"].to_list())
print(countList)
makeBarChart(genderList, countList, "Gender Distribution of Artists on Wikidata", "Number of Artists" )
makePieChart2(genderList, countList, "Percentage of Genders on Wikidata", (0, 0 , 0.5))


other_genders_forBar = artistGenderGrouped[2:].drop_nulls()
makePieChart(other_genders_forBar["genderLabel"].to_list(), other_genders_forBar["artist"].to_list(), "Gender Distribution of 'other' Genders on Wikidata", "coolwarm")

makeTreeMap(other_genders_forBar["genderLabel"].to_list(), other_genders_forBar["artist"].to_list(), "Spectral","Gender Distribution of 'other' Genders on Wikidata", 100 )

horizontalBarChart(other_genders_forBar["genderLabel"].to_list(), other_genders_forBar["artist"].to_list(), "Count of 'other' Genders on Wikidata", "Number of Artists on Wikidata" )

In [ ]:
#Make grouped bar chart
groupBarChart(genderList,countList,countListWiki, "Gender Distribution of Artists on Wikidata and Wikipedia","Number of Artists","Wikidata", "English Wikipedia")

In [ ]:
#Look at the region/country split of the data
regions = artistDF.select(
    pl.col("artist"),
    pl.col("countryCitizenship"),
    pl.col("countryCitizenshipLabel"),
).unique()

print(f"The artists on wikidata cover {len(regions["countryCitizenship"].drop_nulls().unique())} countries (present and historical.")
regionsGrouped = regions.group_by(["countryCitizenship", "countryCitizenshipLabel"]).agg(pl.col("artist").count()).sort(by=pl.col("artist"),descending=True)
regionsGroupedFRAC = regions["countryCitizenship"].drop_nulls().value_counts(sort=True, normalize=True, name="fraction")
display(regionsGroupedFRAC)
regionsGrouped.write_csv("./Visuals Data/artistsGroupedByRegions-Countries.csv", null_value="null/unknown")
display(regionsGrouped.head(5))

countRegionsNull = artistDF.filter(
    pl.col("countryCitizenship").is_null()
).select(pl.col("artist")).unique()

countRegionsNotNull = artistDF.filter(
    pl.col("countryCitizenship").is_not_null()
).select(pl.col("artist")).unique()
print(f"There are {len(countRegionsNull)} artists without a region/country")
print(f"There are {len(countRegionsNotNull)} artists with a region/country")

makePieChart(["With Their Country", "Without Their Country"],[len(countRegionsNotNull),len(countRegionsNull)],"Artists With and Without Their Country of Citizenship on Wikidata", "hls")


In [ ]:
#Bar chart of top 10 countries
top10Regions = regionsGrouped.drop_nulls()[0:10]
horizontalBarChart(top10Regions["countryCitizenshipLabel"].to_list(), top10Regions["artist"].to_list(), "Top 10 Countries for Artists on Wikidata", "Number of Artists on Wikidata" )


In [ ]:
#top 10 regions on wikipedia

regionsWiki = artistDF.filter(pl.col("article").is_not_null()).select(
    pl.col("artist"),
    pl.col("countryCitizenship"),
    pl.col("countryCitizenshipLabel"),
).unique()

print(f"The artists on Wikipedia cover {len(regionsWiki["countryCitizenship"].drop_nulls().unique())} countries (present and historical.")
regionsGroupedWiki = regionsWiki.group_by(["countryCitizenship", "countryCitizenshipLabel"]).agg(pl.col("artist").count()).sort(by=pl.col("artist"),descending=True)
regionsGroupedFRACWiki = regionsWiki["countryCitizenship"].drop_nulls().value_counts(sort=True, normalize=True, name="fraction")
display(regionsGroupedFRACWiki)
regionsGroupedWiki.write_csv("./Visuals Data/wikipediaArtistsGroupedByRegions-Countries.csv", null_value="null/unknown")
display(regionsGroupedWiki.head(5))

top10Regions = regionsGroupedWiki.drop_nulls()[0:10]
horizontalBarChart(top10Regions["countryCitizenshipLabel"].to_list(), top10Regions["artist"].to_list(), "Top 10 Countries for Artists on Wikipedia", "Number of Artists on Wikipedia" )


In [ ]:
countRegionsNullWiki = artistDF.filter(
    pl.col("countryCitizenship").is_null(),
    pl.col("article").is_not_null(),
).select(pl.col("artist")).unique()

countRegionsNotNullWiki = artistDF.filter(
    pl.col("countryCitizenship").is_not_null(),
    pl.col("article").is_not_null()
).select(pl.col("artist")).unique()
makePieChart(["With Their Country", "Without Their Country"],[len(countRegionsNotNullWiki),len(countRegionsNullWiki)],"Artists With and Without Their Country of Citizenship on Wikipedia", "hls")

In [ ]:
import geopandas as gpd
import pyarrow as pa

In [ ]:
#Get the pandas data of countries with their iso A code

#This code was createdwith help from a tutorial from Towards data science accessed 2nd August from: https://towardsdatascience.com/making-heat-maps-with-literal-maps-how-to-use-python-to-construct-a-chloropleth-6b65e4e33905/
countriesDF = pd.read_csv("./mapData/countriesCoordsISOA.csv")
countriesDF["country"] = countriesDF["country"].map(getEntityIDValue)
worldMap = gpd.read_file("./mapData/ne_50m_admin_0_countries/ne_50m_admin_0_countries.shp")
worldMap = worldMap[['ISO_A3','geometry']]
countriesDF[['country', 'ISO_A3']] = countriesDF[['country', 'isoCode']]

groupedRegionsPandas = regionsGroupedWiki.to_pandas()
countriesDF = countriesDF.merge(groupedRegionsPandas, left_on="country", right_on="countryCitizenship", how="inner")

countriesDF = countriesDF[['country', 'ISO_A3','artist']]
countriesDF = countriesDF.groupby(["country",'ISO_A3']).sum()

#print(countriesDF[countriesDF["country"] == "Q30"])
preppedCountryData = worldMap.merge(countriesDF, on='ISO_A3', how='inner')

display(preppedCountryData)
preppedCountryData.plot(column="artist", cmap="YlOrRd", linewidth=4)

countriesDF

In [ ]:
#Look at the genre and gender-genre split on wikidata
genresDF = artistDF.select(
    pl.col("artist"),
    pl.col("genderLabel"),
    pl.col("genre"),
    pl.col("genreLabel"),
).unique()
genresDFGrouped = genresDF.group_by(["genre", "genreLabel"]).agg(pl.col("artist").count()).sort(by=pl.col("artist"), descending=True)
genresGroupBYFRAC = genresDF["genre"].drop_nulls().value_counts(sort=True, normalize=True, name="fraction")
display(genresDFGrouped)
display(genresGroupBYFRAC)
genresDFGrouped.write_csv("./Visuals Data/artistsGroupedByGenres.csv", null_value="null/unknown")
display(genresDFGrouped.sample(5))
genreGenderSplit = genresDF.group_by(["genre", "genreLabel","genderLabel"]).agg(pl.col("artist").count())
print(genreGenderSplit.head())
genreGenderPivot = genreGenderSplit.pivot(
    "genderLabel",
    index = ["genre", "genreLabel"],
    values = "artist",
    aggregate_function = "sum"
)
genreGenderPivot.write_csv("./Visuals Data/artistsGroupedByGenresAndGender.csv", null_value="null/unknown")


countGenreNull = artistDF.filter(
    pl.col("genre").is_null()
).select(pl.col("artist")).unique()

countGenreNotNull = artistDF.filter(
    pl.col("genre").is_not_null()
).select(pl.col("artist")).unique()
print(f"There are {len(countGenreNull)} artists without a genre")
print(f"There are {len(countGenreNotNull)} artists with a genre")
makePieChart(["With Genre", "Without Genre"], [len(countGenreNotNull), len(countGenreNull)], "Artists With and Without Genres on Wikidata", "Paired")
genreGenderPivot


In [ ]:
countGenreNull = artistDF.filter(
    pl.col("genre").is_null(),
    pl.col("article").is_not_null()
).select(pl.col("artist")).unique()

countGenreNotNull = artistDF.filter(
    pl.col("genre").is_not_null(),
    pl.col("article").is_not_null()
).select(pl.col("artist")).unique()
print(f"There are {len(countGenreNull)} artists without a genre on Wikipedia")
print(f"There are {len(countGenreNotNull)} artists with a genre on Wikipedia")
makePieChart(["With Genre", "Without Genre"], [len(countGenreNotNull), len(countGenreNull)], "Artists With and Without Genres on Wikipedia", "Paired")

In [ ]:
#Top Genres on Wikidata for artist

genresWikidata = (artistDF
                  .select(
                        pl.col(["artist", "genreLabel", "genre"])).unique())
makeTop10(genresWikidata, "artist", "genre", "genreLabel", "Wikidata", "Top 10 Genres for Artists on Wikidata")

In [ ]:
#Top 10 Genres on Wikipedia for artists

genresWikipedia = (artistDF
          .filter(pl.col("article").is_not_null())
          .select(
    pl.col(["artist","genre","genreLabel"])
).unique())

makeTop10(genresWikipedia, "artist", "genre", "genreLabel", "Wikipedia", "Top 10 Genres for Artists on Wikipedia")



In [ ]:
#What are the different occupations
len(artistDF.select(pl.col("occupation")).drop_nulls().unique())

In [ ]:
#Look at the occupation and occupation/gender split
occupationDF = artistDF.select(
    pl.col("artist"),
    pl.col("genderLabel"),
    pl.col("occupation"),
    pl.col("occupationLabel"),
).unique()
occupationDFDFGrouped = occupationDF.group_by(["occupation", "occupationLabel"]).agg(pl.col("artist").count())
occupationDFDFGrouped.write_csv("./Visuals Data/artistsGroupedByOccupation.csv", null_value="null/unknown")
display(occupationDFDFGrouped.sample(5))
occupationGenderSplit = occupationDF.group_by(["occupation", "occupationLabel","genderLabel"]).agg(pl.col("artist").count())

occupationGenderPivot = occupationGenderSplit.pivot(
    "genderLabel",
    index = ["occupation", "occupationLabel"],
    values = "artist",
    aggregate_function = "sum"
)

occupationGenderPivot.write_csv("./Visuals Data/artistsGroupedByOccupationAndGender.csv", null_value="null/unknown")
display(occupationGenderPivot.sample(5))

Old Code that I will ignore for now

In [ ]:
cleanedArtistDF = artistDF.filter(
    ~pl.col("artist").is_in(pl.col("artistLabel").implode()),
    ~pl.col("influencedBY").is_in(pl.col("influencedBYLabel").implode()),
    ~pl.col("recordLabel").is_in(pl.col("recordLabelLabel").implode()),
    ~pl.col("genre").is_in(pl.col("genreLabel").implode()),
)

#Add a column for presence of wikiPage
cleanedArtistDF = cleanedArtistDF.with_columns(
    isThereAWiki = pl.col("article").is_not_null()
)
#Finally de-duplicate the data.
cleanedArtistDF = cleanedArtistDF.unique()
display(cleanedArtistDF.head(5))
display(cleanedArtistDF.shape)

That is the end of the cleaning stage. Now time to analyse the data. Null values were left intentionally as we are looking for the gaps

In [ ]:
#Discover the gender gap in the set.
artistsByGender = cleanedArtistDF.select(
    pl.col("artist"),
    pl.col("gender"),
    pl.col("genderLabel"),
)
artistsByGender = artistsByGender.unique()
artistGenderGrouped = artistsByGender.group_by(["gender", "genderLabel"]).agg(pl.col("artist").count())
artistGenderGroupedFRAC = artistsByGender["genderLabel"].value_counts(sort=True, normalize=True, name="fraction")
display(artistGenderGrouped)
display(artistGenderGroupedFRAC)

In [ ]:
genderChart = (
    alt.Chart(artistGenderGrouped).mark_bar().encode(
    y="artist",
    x="genderLabel",
    )
    .properties(width=500, title="Gender Distribution of Wikidata")
)
genderChart.encoding.x.title = "Genders"
genderChart.encoding.y.title = "Counts"
genderChart.show()



genderBase = alt.Chart(artistGenderGrouped).encode(
    alt.Theta("artist:Q").stack(True),
    alt.Color("genderLabel:N").legend(None)
    )
pie = genderBase.mark_arc(outerRadius=120)
text = genderBase.mark_text(radius=140, size=20).encode(text="genderLabel:N")
pie+text

In [ ]:
#Top genres and genre gap

In [ ]:
#Top occupations

In [ ]:
#Top occupations by gender

In [ ]:
#Gender and genre split